# Is the companion's final model better than the chain ladder?

This notebook scores the second tlrn study. The question is simple: does the companion
study's newest model (13 features, a factor head, attention across lines and lags, blended
with the multivariate chain ladder, 20 members averaged) have a lower reserve error than the
chain ladder (`mack`) and the multivariate chain ladder (`mcl`)?

The model was trained on a Colab GPU with `scripts/tlrn_study_colab.py --study factor_head`:
ten independent fits (draws) of each row on each cohort. This notebook only **reads the saved
fits**. It fits the two benchmarks itself, on the cells known at each of four valuation dates,
and runs the evaluator in `scripts/tlrn_study2_eval.py`. The rules for what counts as
`effective` were written down before any fit existed, in
`knowledge/decisions/tlrn-factor-head-study.md`; the evaluator's docstring repeats them. This
notebook does not change them.

Five things to keep in mind while reading:

* **Cohorts A and B are not out-of-sample.** The companion chose its design and its
  hyperparameters with their test data in view, so a verdict on A or B says only whether that
  fixed specification, run many times, beats the benchmarks on the data it was tuned on.
* **Cohort C (accident years 1988 to 1997) is out-of-sample in accident years only.** As far
  as we know the companion did not tune on it (assumed, not known). Its outcomes sit in
  statement years 1998 to 2006, which overlap the data of A and B and the companion's
  valuations 6, 7 and 8, and 67 of its 83 companies are in A and 60 in B (155 of its 232 pairs
  are B's pairs). So it does not test a new economic environment or other companies. Every
  cohort is also selected on future cells existing (a complete 10-by-10 square, outcome cells
  included), a survivorship that applies to the benchmarks as well as the model. Read cohort C
  first: if its verdict is `not distinguishable` or `worse`, the main question is answered
  and A and B are confirmation runs (Amendment 1 of the decision note).
* **The primary verdict cannot detect a small gain.** In the first study the paired 95%
  intervals on A and B were 5.4 to 11.0 points of Pool_APE wide (9.3 to 11.0 against `mack`,
  5.4 to 7.8 against `mcl`), so a gain has to be about three points or more to clear both benchmarks. A gain of
  about half a point, the size of the companion's headline gain, cannot be detected, and `not
  distinguishable` on A or B is the expected result, not news.
* **A second question is asked beside the verdict:** does the network add anything beyond the
  blend? It compares the main row with the factors-only row, which share data, factors and
  blend, and it is reported separately; it never changes the verdict.
* **A verdict is provisional** unless the folder holds exactly draws 0 to 9 of the main row.

Every sentence below that states a result is written from the computed values, so it says
nothing until the fits exist.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd().resolve()
REPO = next(p for p in (HERE, *HERE.parents) if (p / "scripts" / "tlrn_study2_eval.py").exists())
sys.path.insert(0, str(REPO / "scripts"))

import tlrn_study2_eval as ev  # noqa: E402
import tlrn_study_colab as study  # noqa: E402

# cohort C first: it is the cohort whose verdict decides the main question
COHORT_ORDER = ("C", "A", "B")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# Where the saved fits are: the study2 folder downloaded from Drive (My Drive/ibnr_colab_fh/study2)
# to C:\Temp\study2. Set TLRN_STUDY2_FITS to use another path.
FITS_DIR = Path(os.environ.get("TLRN_STUDY2_FITS", r"C:\Temp\study2"))
# The benchmark fits take a minute or two per cohort; they are kept here and reused.
CACHE_DIR = Path(os.environ.get("TLRN_STUDY2_CACHE", r"C:\Temp\study2_cache"))
# Optional: the companion's rolling table (R_T1_rolling_origin.csv), to tie the benchmarks out.
COMPANION_ROLLING = os.environ.get("TLRN_COMPANION_ROLLING")

if not FITS_DIR.exists():
    raise SystemExit(
        f"No folder of saved fits at {FITS_DIR}. Copy the Colab run's study2 folder there, or "
        "set TLRN_STUDY2_FITS to its path."
    )

# Read the records: which are this study's, and are they smoke runs?
import json  # noqa: E402

records = {}
for path in sorted(FITS_DIR.glob("*-ibnr*.json")):
    if path.name.startswith("manifest"):
        continue
    info = json.loads(path.read_text(encoding="utf-8"))
    s = info.get("settings", {})
    if s.get("study") == 2 and s.get("study_name") == "factor_head":
        records[path.name] = info
SMOKE = bool(records) and all(r["settings"].get("smoke") for r in records.values())
BANNER = (
    "SMOKE RUN. The fits were a few epochs and a few members. Nothing below may be quoted."
    if SMOKE
    else ""
)
print(f"{len(records)} finished fits of the second study in {FITS_DIR}")
if BANNER:
    print(BANNER)

## 1. What is in the folder

One row per cohort and row: how many draws are saved and which, what they ran on, and how
long a fit took. The registered study is draws 0 to 9 of both rows on each of A, B and C.

In [ ]:
rows = []
for cohort_name in COHORT_ORDER:
    for row in ev.ROWS:
        found = [
            r
            for r in records.values()
            if r["settings"]["cohort"] == cohort_name
            and r["settings"]["row"] == row
            and bool(r["settings"].get("smoke")) == SMOKE
        ]
        draws = sorted(r["settings"]["draw"] for r in found)
        seconds = [r["run"].get("fit_seconds") for r in found if r["run"].get("fit_seconds")]
        devices = sorted({str(r["run"].get("device_kind", "?")) for r in found})
        rows.append(
            {
                "cohort": cohort_name,
                "row": row,
                "draws": len(draws),
                "which": ",".join(map(str, draws)) or "none",
                "complete": draws == list(range(10)),
                "device": "; ".join(devices) or "-",
                "median_fit_seconds": float(np.median(seconds)) if seconds else np.nan,
            }
        )
inventory = pd.DataFrame(rows)
inventory

## 2. Scoring the fits

For each cohort with fits, the evaluator (`ev.run_cohort`) does four things.

1. It builds the cohort from the cas-schedule-p mart and checks its counts and hash.
2. It fits `mack` (one fit per company-line pair) and `mcl` (one fit per company) with ibnr's
   gallery entries on the cells known at each valuation date, and reads their forecasts cell
   by cell, so they are scored on exactly the cells the transformer is scored on.
3. It checks that each saved fit's tables agree with its own cell sums: the number of cells per
   company and date, the known actuals, and the valuation-10 forecast against the stored
   reserve. A fit that fails is refused by name.
4. It computes the pre-registered statistics.

A valuation date `v` means calendar diagonal `v` of the 10-by-10 square; 10 is the last known
one. At `v` below 10 the cells scored are the later known diagonals at development lag `v` or
less, and the actual is the known paid over them. At 10 the cells are every cell past the last
diagonal and the actual is the realised reserve. The benchmark fits are cached in `CACHE_DIR`.

In [ ]:
results, cohorts = {}, {}
cells_ab = None
for cohort_name in COHORT_ORDER:
    if cohort_name in ("A", "B"):
        cells_ab = cells_ab if cells_ab is not None else study.cells_from_mart()
        cells = cells_ab
    else:
        cells = study.cells_from_mart(years=study.YEARS[cohort_name])
    got = ev.run_cohort(
        cohort_name,
        FITS_DIR,
        smoke=SMOKE,
        cells=cells,
        cache_dir=CACHE_DIR,
        companion_rolling=COMPANION_ROLLING,
    )
    if got is not None:
        cohorts[cohort_name], results[cohort_name] = got
print("\nscored cohorts:", list(results) or "none")

## 3. Do the benchmarks and the scoring window tie out?

Before any model number is read, the numbers it is compared with have to be right. The
checks printed here come from the evaluator:

* on cohort A, `mack` and `mcl` must reproduce notebook 04's valuation-10 Pool_APE (the
  evaluator stops if they do not);
* when the companion's rolling table is supplied (`TLRN_COMPANION_ROLLING`), its cell and
  company counts must equal ours at all four dates, and its chain ladder and chain ladder +
  MCL Pool_APE are shown beside `mack` and `mcl`.

In [ ]:
for name, res in results.items():
    print(f"=== cohort {name}: {cohorts[name].description}")
    print(f"scored companies: {len(res['companies'])} of the board {len(cohorts[name].board)}")
    if res["left_out"]:
        print("left out because a benchmark failed:", res["left_out"])
    for key, value in res["extras"].items():
        if key == "companion's rolling table":
            continue
        print(f"{key}: {ev.fmt_extra(value)}")
    if "companion_rolling" in res:
        table = res["companion_rolling"]
        display(table.round(6))
        print(
            "largest difference from the companion's Pool_APE:",
            f"mack {table['mack_difference'].abs().max():.1e},",
            f"mcl {table['mcl_difference'].abs().max():.1e}",
        )
    print(
        "cells scored by valuation:",
        {v: res["n_cells_by_valuation"][v] for v in ev.VALUATIONS},
    )
    print()
if not results:
    print("No cohort has finished fits in the folder yet.")

## 4. The verdicts

The rule, for the main row `tlrn_13_fh` on each cohort, from the draw average (every draw's
company forecasts averaged, then scored). It is `effective` only if all four hold:

1. at valuation 10, the Pool_APE difference (model minus benchmark) has a 95% interval wholly
   below zero against both `mack` and `mcl`;
2. the mean over the four valuation dates of that difference has an interval wholly below
   zero against both;
3. the median over the draws of the valuation-10 Pool_APE is below both benchmarks';
4. with the dominant company left out, the valuation-10 interval is not wholly above zero
   against either benchmark.

It is `worse` if the interval in (1) or (2) is wholly above zero against either benchmark,
and `not distinguishable` otherwise. The intervals come from 10,000 resamples of companies,
with each company's four valuation dates kept together.

In [ ]:
table = []
for name, res in results.items():
    row = {
        "cohort": name,
        "what a verdict here means": res["status"],
        "draws": res["n_draws"],
        "verdict": res["verdict_label"],
    }
    if res.get("conditions"):
        for key, value in res["conditions"].items():
            row[key] = "yes" if value else "no"
    table.append(row)
verdict_table = pd.DataFrame(table)
display(verdict_table)

for name, res in results.items():
    if res["verdict"] is None:
        print(f"Cohort {name}: there are no finished fits of {ev.MAIN_ROW}, so no verdict is made.")
        continue
    print(f"Cohort {name} ({res['status']}):")
    print(f"  the pre-registered rule gives '{res['verdict_label']}'.")
    print(f"  {res['verdict_reason']}")
    if res["provisional"]:
        print("  This is provisional: the registered study is exactly draws 0 to 9.")
    print()

## 4b. A second question: does the network add anything?

Amendment 1 of the decision note adds one paired question, reported beside the verdict and
never part of it. The main row and the factors-only row (the head's pooled factors with the
blend weight, network off) have the same cohort, companies and draws, and both are blended
with the multivariate chain ladder, so their errors are highly correlated and most of the
company-to-company noise cancels in their difference. The statistic is the Pool_APE of the
draw average, main row minus factors-only row, with the same 10,000-resample bootstrap. The
word is `network adds value` only if the valuation-10 interval and the mean-over-dates
interval are both wholly below zero and, with the dominant company left out, the
valuation-10 interval is not wholly above zero; `network hurts` if either of the first two is
wholly above zero; `not distinguishable` otherwise.

In [ ]:
net = []
for name, res in results.items():
    row = {
        "cohort": name,
        "network question": res["network_label"],
    }
    for key, value in (res.get("network_conditions") or {}).items():
        row[key] = "yes" if value else "no"
    net.append(row)
display(pd.DataFrame(net))

for name, res in results.items():
    print(f"Cohort {name}: {res['network_reason']}")
    if len(res["network_check"]):
        display(res["network_check"].round(5))
    print()

## 5. How much one draw varies

Each dot is the Pool_APE of one draw (one fit of 20 members) of the main row; the cross is the
Pool_APE of the draw average, and the two horizontal marks are `mack` and `mcl`. If the dots
spread across both marks, then a single fit's rank against the benchmarks depends on the
draw, and the draw average is the fairer thing to judge.

In [ ]:
def valuation_labels():
    return [str(v) for v in ev.VALUATIONS] + ["rolling"]


plotted = [n for n, r in results.items() if ev.MAIN_ROW in r["draws"]]
if plotted:
    fig, axes = plt.subplots(1, len(plotted), figsize=(5.2 * len(plotted), 4), squeeze=False)
    for ax, name in zip(axes[0], plotted, strict=True):
        res = results[name]
        draws = res["draw_pool_ape"]
        draws = draws[draws["row"] == ev.MAIN_ROW]
        columns = [f"valuation_{v}" for v in ev.VALUATIONS] + ["rolling"]
        for i, column in enumerate(columns):
            ax.scatter(np.full(len(draws), i), draws[column], s=18, color="#4c78a8", alpha=0.7)
        average = ev.pool_ape_by_valuation(res["series"][ev.MAIN_ROW], res["actual"])
        average = np.append(average, ev.rolling(average))
        ax.scatter(
            range(len(columns)), average, marker="x", color="black", s=50, label="draw average"
        )
        for b, color in (("mack", "#e45756"), ("mcl", "#54a24b")):
            pools = res["benchmark_pool_ape"][b]
            level = np.append(pools, ev.rolling(pools))
            ax.hlines(
                level,
                np.arange(len(columns)) - 0.3,
                np.arange(len(columns)) + 0.3,
                color=color,
                lw=2.5,
                label=b,
            )
        ax.set_xticks(range(len(columns)), valuation_labels())
        ax.set_xlabel("valuation date")
        ax.set_ylabel("Pool_APE")
        ax.set_title(f"cohort {name}: {res['n_draws']} draws" + (" (SMOKE)" if SMOKE else ""))
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print("No cohort has finished fits of the main row yet.")

## 6. The size of the difference, with intervals

Each point is the Pool_APE of the draw average minus the benchmark's, with its 95% bootstrap
interval. Negative favours the model. The last line of each group is the valuation-10
difference with the dominant company left out. An interval that does not reach zero is the
only kind the verdict rule trusts.

In [ ]:
def forest(res, benchmark):
    d = res["differences"]
    d = d[(d["method"] == ev.MAIN_ROW) & (d["benchmark"] == benchmark)].set_index("statistic")
    order = [f"valuation {v}" for v in ev.VALUATIONS] + ["rolling"]
    out = d.loc[order, ["diff", "low", "high"]].copy()
    ex = res["excluding_dominant"]
    ex = ex[(ex["method"] == ev.MAIN_ROW) & (ex["benchmark"] == benchmark)].iloc[0]
    out.loc["valuation 10, without the dominant company"] = [ex["diff"], ex["low"], ex["high"]]
    return out


if plotted:
    fig, axes = plt.subplots(len(plotted), 2, figsize=(11, 3.4 * len(plotted)), squeeze=False)
    for r, name in enumerate(plotted):
        for c, b in enumerate(ev.BENCHMARKS):
            ax = axes[r][c]
            data = forest(results[name], b)
            y = np.arange(len(data))[::-1]
            ax.errorbar(
                data["diff"],
                y,
                xerr=[data["diff"] - data["low"], data["high"] - data["diff"]],
                fmt="o",
                color="#4c78a8",
                capsize=3,
            )
            ax.axvline(0, color="black", lw=1)
            ax.set_yticks(y, data.index, fontsize=8)
            ax.set_xlabel("Pool_APE, model minus " + b)
            ax.set_title(f"cohort {name} against {b}" + (" (SMOKE)" if SMOKE else ""), fontsize=10)
    fig.tight_layout()
    plt.show()
    for name in plotted:
        for b in ev.BENCHMARKS:
            print(f"cohort {name} against {b}")
            display(forest(results[name], b).round(5))

## 7. With and without the dominant company

In these cohorts one company holds roughly two thirds of the reserve (the next cell prints the exact share), so a Pool_APE that looks
good or bad may be that one company. The table gives the valuation-10 Pool_APE of the
model, the raw network, and both benchmarks with every company and with the dominant one
left out. The companion study reports both versions too.

In [ ]:
def pool_without_dominant(res):
    last = ev.VALUATIONS.index(ev.LAST)
    keep = np.array([c != res["dominant_company"] for c in res["companies"]])
    rows = {}
    for method, pred in res["series"].items():
        rows[method] = {
            "all companies": ev.pool_ape(pred[last], res["actual"][last]),
            "without the dominant company": ev.pool_ape(
                pred[last][keep], res["actual"][last][keep]
            ),
        }
    return pd.DataFrame(rows).T


for name, res in results.items():
    print(
        f"cohort {name}: the dominant company is {res['dominant_company']}, with "
        f"{res['dominant_share']:.1%} of the valuation-{ev.LAST} reserve"
    )
    display(pool_without_dominant(res).round(5))

## 8. What the network adds

Three versions of the same forecast, from the same fits: the blended model (the main row),
the raw network before the blend with `mcl`, and the factors-only row (the head's pooled
factors and the blend weight, with the network switched off). If the main row is no better
than the factors-only row, the network is adding nothing beyond the head and the blend. The
last table gives the difference from the main row with its interval (positive means the main
row is better). The pre-registered paired question, with its own rule, is section 4b.

In [ ]:
order = [ev.MAIN_ROW, ev.FACTORS_ROW, ev.RAW_NETWORK, "mack", "mcl"]
if results:
    fig, axes = plt.subplots(1, len(results), figsize=(5.2 * len(results), 3.8), squeeze=False)
    for ax, (name, res) in zip(axes[0], results.items(), strict=True):
        present = [m for m in order if m in res["series"]]
        last = ev.VALUATIONS.index(ev.LAST)
        v10 = [ev.pool_ape(res["series"][m][last], res["actual"][last]) for m in present]
        roll = [
            ev.rolling(ev.pool_ape_by_valuation(res["series"][m], res["actual"])) for m in present
        ]
        x = np.arange(len(present))
        ax.bar(x - 0.2, v10, 0.4, label="valuation 10", color="#4c78a8")
        ax.bar(x + 0.2, roll, 0.4, label="mean of the four dates", color="#f58518")
        ax.set_xticks(x, present, rotation=30, ha="right", fontsize=8)
        ax.set_ylabel("Pool_APE")
        ax.set_title(f"cohort {name}" + (" (SMOKE)" if SMOKE else ""), fontsize=10)
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
    for name, res in results.items():
        print(f"cohort {name}: each row minus the main row")
        display(res["ablations"].round(5) if len(res["ablations"]) else "no comparison rows yet")

## 9. The other metrics

Reported beside the verdict, not part of it. `pool_pe` is the bias (sum of errors over sum
of actuals); `medape` the median absolute percentage error; `mae` the mean absolute error;
`log_ratio` the mean absolute log of predicted over actual reserve, over companies where both
are positive; `share_better_than_*` the share of companies where the row's absolute error is
smaller than the benchmark's. The mean absolute percentage error (`mape`) and the
reserve-weighted RMSE (`wrmse`) are **descriptive only**: they are driven by very small
reserves and by one large company, which is where the companion's claimed gains are largest.

In [ ]:
for name, res in results.items():
    m = res["metrics"]
    m = m[m["valuation"].isin(["10", "rolling"])]
    print(f"cohort {name}: reported metrics")
    display(
        m[
            [
                "method",
                "valuation",
                "pool_ape",
                "pool_pe",
                "medape",
                "mae",
                "log_ratio",
                "share_better_than_mack",
                "share_better_than_mcl",
            ]
        ]
        .round(5)
        .reset_index(drop=True)
    )
    print(f"cohort {name}: descriptive only")
    display(m[["method", "valuation", "mape", "wrmse"]].round(5).reset_index(drop=True))

## 10. By size of reserve, and the blend weight

Pool_APE within quartiles of the actual reserve at valuation 10 (quartile 4 holds the
largest reserves, and the dominant company), then the median blend weight `alpha` over all
members and draws at each valuation date. The companion's published median `alpha` is around
0.63 at valuation 10.

In [ ]:
for name, res in results.items():
    q = res["quartiles"]
    q = q[q["valuation"] == ev.LAST]
    print(f"cohort {name}: Pool_APE by quartile of actual reserve, valuation {ev.LAST}")
    display(q.pivot_table(index=["quartile", "n"], columns="method", values="pool_ape").round(4))
    if "alpha" in res:
        print(f"cohort {name}: median blend weight")
        display(res["alpha"].round(4))

## What this cannot show

* The companion's design was tuned on A and B. Cohort C is independent in accident years but
  not in calendar time (its outcomes sit in statement years 1998 to 2006, overlapping A and B
  and the companion's valuations 6, 7 and 8) and not in companies (67 of its 83 companies are
  in A, 60 in B). Whether the companion tuned on C is not known: it is assumed it did not. So
  C does not test a new economic environment or other companies.
* Every cohort is selected on future cells existing, which is a shared survivorship: it applies
  to the benchmarks as well as the model.
* The primary verdict on one cohort can detect a gain of about three points or more of Pool_APE
  and not a half-point gain, so `not distinguishable` is the expected result.
* The bootstrap resamples companies. It measures company-to-company sampling and says
  nothing about other years beyond what cohort C adds.
* Ten draws sample training randomness thinly, and a GPU rerun gives different draws.
* The model blends the multivariate chain ladder into every member (the weight is fitted),
  so it is partly the benchmark it is compared with.
* Three cohorts are judged and no multiplicity correction is made.